In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point

In [3]:
df = pd.read_csv('/content/drive/MyDrive/final_csv/원본 파일/Chicago_Crimes_2008_to_2011.csv', on_bad_lines='skip')

In [4]:
df = df.dropna(subset=['X Coordinate','Y Coordinate','Latitude', 'Longitude', 'Location'])

In [5]:
df.isna().sum()

,0
Unnamed: 0,0
ID,0
Case Number,6
Date,0
Block,0
IUCR,0
Primary Type,0
Description,0
Location Description,233
Arrest,0


In [6]:
# 3. 범죄 데이터의 위도, 경도 정보를 사용해 GeoDataFrame 생성
df['geometry'] = df.apply(lambda row: Point(row['Longitude'], row['Latitude']), axis=1)
crime_gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")  # WGS84 좌표계 설정

In [7]:
# 4. Community Area CSV 파일 불러오기 (the_geom을 WKT 형식으로 변환)
community_areas = pd.read_csv('/content/drive/MyDrive/final_csv/CommAreas_20250325.csv')  # Community Area 데이터 (CSV)
community_areas['geometry'] = community_areas['the_geom'].apply(wkt.loads)  # the_geom을 WKT로 변환
community_areas_gdf = gpd.GeoDataFrame(community_areas, geometry='geometry', crs="EPSG:4326")

In [8]:
# 5. Ward CSV 파일 불러오기 (the_geom을 WKT 형식으로 변환)
wards = pd.read_csv('/content/drive/MyDrive/final_csv/Wards_2003_2015.csv')  # Ward 데이터 (CSV)
wards['geometry'] = wards['the_geom'].apply(wkt.loads)  # the_geom을 WKT로 변환
wards_gdf = gpd.GeoDataFrame(wards, geometry='geometry', crs="EPSG:4326")

In [9]:
wards_gdf = wards_gdf[wards_gdf['WARD'] != 'OUT']

In [10]:
wards_gdf= wards_gdf.dropna(subset=['WARD'])

# int로 변환 후 float로 변환
wards_gdf['WARD'] = wards_gdf['WARD'].astype(float)

In [11]:
wards_gdf['WARD'].unique()

array([ 4., 33., 49., 37., 18., 31., 25.,  8., 26., 28.,  3., 47.,  1.,
       38., 11., 30., 39., 12.,  9.,  6.,  5., 19., 41., 23., 24., 46.,
       44., 36., 48., 27., 50.,  7., 15., 34., 40., 10.,  2., 22., 35.,
       32., 17., 21., 16., 45., 42., 13., 14., 43., 29., 20.])

In [12]:
# 6. 범죄 데이터의 좌표가 어느 Ward에 속하는지 Spatial Join
crime_with_ward = gpd.sjoin(crime_gdf, wards_gdf[['WARD', 'geometry']], how='left', predicate='within')  # Ward 매핑

In [13]:
# 7. 범죄 데이터의 좌표가 어느 Community Area에 속하는지 Spatial Join
crime_with_community = gpd.sjoin(crime_with_ward, community_areas_gdf[['AREA_NUMBE', 'geometry']], how='left', predicate='within', lsuffix='_ward', rsuffix='_community')  # Community Area 매핑

In [14]:
# 8. 'Ward'와 'Community Area' 결측치 채우기
crime_with_community['Ward'] = crime_with_community['Ward'].fillna(crime_with_community['WARD'])
crime_with_community['Community Area'] = crime_with_community['Community Area'].fillna(crime_with_community['AREA_NUMBE'])

In [15]:
# 9. 불필요한 열 삭제 (매핑된 'WARD', 'COMMUNITY' 열 제거)
crime_with_community = crime_with_community.drop(columns=['WARD', 'index_right','index__community', 'AREA_NUMBE'])

In [16]:
crime_with_community = crime_with_community.drop(columns=['geometry'])

In [17]:
# 10. 결측치 확인
print(crime_with_community[['Ward', 'Community Area']].isna().sum())  # 결측치 확인

Ward              55
Community Area    12
dtype: int64


In [18]:
crime_with_community.isna().sum()

,0
Unnamed: 0,0
ID,0
Case Number,6
Date,0
Block,0
IUCR,0
Primary Type,0
Description,0
Location Description,233
Arrest,0


In [19]:
crime_with_community.head()

,Unnamed: 0,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,388,4785,HP610824,10/07/2008 12:39:00 PM,000XX E 75TH ST,0110,HOMICIDE,FIRST DEGREE MURDER,ALLEY,True,...,6.0,69.0,01A,1178207.0,1855308.0,2008,08/17/2015 03:03:40 PM,41.758276,-87.622451,"(41.758275857, -87.622451031)"
1,835,4786,HP616595,10/09/2008 03:30:00 AM,048XX W POLK ST,0110,HOMICIDE,FIRST DEGREE MURDER,STREET,True,...,24.0,25.0,01A,1144200.0,1895857.0,2008,08/17/2015 03:03:40 PM,41.870252,-87.746069,"(41.87025207, -87.746069362)"
2,1334,4787,HP616904,10/09/2008 08:35:00 AM,030XX W MANN DR,0110,HOMICIDE,FIRST DEGREE MURDER,PARK PROPERTY,False,...,18.0,66.0,01A,1157314.0,1859778.0,2008,08/17/2015 03:03:40 PM,41.770990,-87.698901,"(41.770990476, -87.698901469)"
3,1907,4788,HP618616,10/10/2008 02:33:00 AM,052XX W CHICAGO AVE,0110,HOMICIDE,FIRST DEGREE MURDER,RESTAURANT,False,...,37.0,25.0,01A,1141065.0,1904824.0,2008,08/17/2015 03:03:40 PM,41.894917,-87.757358,"(41.894916924, -87.757358147)"
4,2436,4789,HP619020,10/10/2008 12:50:00 PM,026XX S HOMAN AVE,0110,HOMICIDE,FIRST DEGREE MURDER,GARAGE,False,...,22.0,30.0,01A,1154123.0,1886297.0,2008,08/17/2015 03:03:40 PM,41.843826,-87.709893,"(41.843826272, -87.709893465)"


In [20]:
crime_with_community['Community Area'].unique()

array([69., 25., 66., 30., 28., 31., 43., 63., 67., 44., 49., 27.,  6.,
       23., 40., 56., 26., 20., 21., 29., 61.,  3., 38., 55., 33., 46.,
       51., 68., 52., 71., 45., 19., 42.,  8., 35., 73., 41.,  1., 16.,
       75., 60., 47., 65., 58., 53., 14., 70., 18., 48., 54., 22., 24.,
       34., 57., 50., 39.,  2., 64., 12., 32.,  7., 37.,  4.,  5., 72.,
       10., 15., 74., 36., 17., 59., 77., 13., 76., 11., 62.,  9.,  0.,
       nan])

In [21]:
crime_with_community['Ward'].unique()

array([ 6., 24., 18., 37., 22.,  2., 25.,  5., 16., 14.,  9., 34., 32.,
       27., 20., 15., 23., 30., 35.,  7., 46.,  3., 17.,  1., 10., 28.,
       29.,  8., 31., 49., 33., 21., 11., 13., 39., 12., 36., 19., 26.,
        4., 40., 41., 42., 43., 47., 44., 38., 50., 48., 45., nan])

In [22]:
# 11. 결과를 CSV로 저장
crime_with_community.to_csv('/content/drive/MyDrive/chicago_crime_data_with_ward_community3.csv', index=False)


In [ ]:
crime_with_community[['ID', 'Ward', 'Community Area']]

,ID,Ward,Community Area
1,4676906,11.0,61.0
4,4677901,34.0,49.0
6,4791194,9.0,50.0
7,4679521,21.0,73.0
9,4680124,24.0,29.0
...,...,...,...
1923510,4781176,37.0,19.0
1923511,4671197,38.0,15.0
1923512,4671380,18.0,71.0
1923513,4782588,10.0,46.0
